# 28 — Parallel Subgraph (Send API Map-Reduce)

Fan out N workers in parallel using the LangGraph **Send API**, collect all results with `Annotated[list, operator.add]`, then synthesise in a single aggregate node.

**What you'll learn**
- `Send("node", state)` — dispatch N identical node calls with different per-item inputs
- `Annotated[list, operator.add]` — LangGraph merges parallel workers' outputs automatically
- The map-reduce pattern: `distribute → [N×summarize in parallel] → aggregate`
- Why `aggregate` runs exactly once — only after ALL workers have finished
- How per-worker `doc` state relates to the shared `summaries` fan-in field

**Companion examples:** 26-rag-fusion (Send API for parallel retrieval), 6-multi-agent-graph (multiple agents in one graph)

In [ ]:
# ── 1. Install dependencies (Colab only) ──────────────────────────────────────
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    %pip install -q langchain langchain-openai python-dotenv langgraph

In [ ]:
# ── 2. API key ─────────────────────────────────────────────────────────────────
import os

from dotenv import load_dotenv

if not IN_COLAB:
    load_dotenv()

if IN_COLAB and not os.environ.get("OPENAI_API_KEY"):
    from google.colab import userdata

    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY before running"

In [ ]:
# ── 3. How Annotated[list, operator.add] works ────────────────────────────────
# Normal state field:   state["summaries"] = ["new"]  ← REPLACES old value
# Annotated with add:  state["summaries"] += ["new"]  ← APPENDS to existing
#
# When N workers each return {"summaries": ["one item"]},
# LangGraph calls operator.add to merge all N lists into one.

import operator

existing = ["summary A"]
from_worker_2 = ["summary B"]
merged = operator.add(existing, from_worker_2)
print("operator.add(['summary A'], ['summary B']) =", merged)
# This is exactly what LangGraph does under the hood for each parallel worker

In [ ]:
# ── 4. Build the map-reduce graph ─────────────────────────────────────────────
from operator import add
from typing import Annotated, TypedDict

from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph
from langgraph.types import Send


class State(TypedDict):
    documents: list[str]
    doc: str  # per-worker: set by Send for each branch
    summaries: Annotated[list, add]  # fan-in: all workers append here
    final_summary: str


llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)


def distribute(state: State):
    """Fan-out: return one Send per document.

    All N summarize calls run simultaneously.
    Wall-clock time = slowest single worker, NOT sum of all workers.
    """
    return [Send("summarize", {"doc": doc, "summaries": []}) for doc in state["documents"]]


def summarize(state: State) -> dict:
    """Worker node — runs in parallel for each document."""
    response = llm.invoke(
        [
            SystemMessage("Summarize this in one sentence."),
            HumanMessage(state["doc"]),
        ]
    )
    return {"summaries": [response.content]}


def aggregate(state: State) -> dict:
    """Fan-in: called ONCE after ALL parallel workers complete.

    state['summaries'] already contains all N items, merged by operator.add.
    """
    combined = "\n".join(f"- {s}" for s in state["summaries"])
    response = llm.invoke(
        [
            SystemMessage("Synthesise these summaries into two sentences."),
            HumanMessage(combined),
        ]
    )
    return {"final_summary": response.content}


graph = StateGraph(State)
graph.add_node("summarize", summarize)
graph.add_node("aggregate", aggregate)
graph.add_conditional_edges(START, distribute, ["summarize"])  # fan-out
graph.add_edge("summarize", "aggregate")
graph.add_edge("aggregate", END)
app = graph.compile()

print("Graph: START → [N×summarize in parallel] → aggregate → END")

In [ ]:
# ── 5. Run the map-reduce ─────────────────────────────────────────────────────
SAMPLE_DOCS = [
    "LangGraph is a framework for building stateful, multi-actor LLM applications using graph-based workflows with built-in checkpointing and human-in-the-loop support.",
    "Retrieval-Augmented Generation (RAG) combines a retriever with a language model, grounding answers in external documents to reduce hallucination and improve factuality.",
    "Reciprocal Rank Fusion merges multiple ranked document lists by summing inverse rank scores, rewarding documents that appear high across many different query variants.",
    "Human-in-the-loop checkpointing pauses graph execution at an interrupt node, surfaces state to a human reviewer, and resumes only after an explicit approval decision.",
]

result = app.invoke({"documents": SAMPLE_DOCS, "doc": "", "summaries": [], "final_summary": ""})

print(f"Per-document summaries ({len(result['summaries'])} workers ran in parallel):")
for i, s in enumerate(result["summaries"], 1):
    print(f"  [{i}] {s}")

print(f"\nFinal synthesis:\n  {result['final_summary']}")

## Exercises

**Exercise 1 — Scale up:** Extend `SAMPLE_DOCS` to 8 or 10 items. The Send API fans them all out at once — watch how total time grows compared to a sequential loop.

**Exercise 2 — Change the worker task:** Replace the summarise prompt with `"Extract three keywords"`. What does `aggregate` receive? Update it to handle a list of keyword strings.

**Exercise 3 — Check ordering:** Run the graph twice. Are `result['summaries']` always in the same order as `SAMPLE_DOCS`? What does this tell you about index-dependent logic in parallel workflows?

## What's next

- **26-rag-fusion** — Send API applied to parallel retrieval with RRF merge
- **30-agentic-rag** — agent-driven tool selection (no hard-coded graph edges)
- **6-multi-agent-graph** — multiple specialist agents sharing one state graph